# Week 3 · Module 3 Practical — Rebuild It Native 🔩

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 3 · Module 3: The loop, done properly — native function calling, ReAct, memory, LangChain**

---

| # | Rung | You build |
|---|------|-----------|
| 1 | Schemas | Day 1's tools converted to the API's tools format |
| 2 | The ledger | ONE round trip, raw `tool_calls` objects printed |
| 3 | Native loop | Day 1's 15 lines, hardened — plus **parallel tool calls** |
| 4 | ReAct | Thoughts appear in the trace; compare tool choice |
| 5 | Memory | Cross-turn conversation + safe trimming |
| 6 | X-ray | Same tools in **LangChain** — decoded line by line |
| 🏁 | **AgentV1** | Everything combined, trace and all |

> ⏳ The LangChain install (rung 6) takes ~1 min — a cell early on starts it in the background of your reading.

## Step 0 — Setup 🔑

In [ ]:
%pip install -q -U openai
# start the LangChain install now too — used in rung 6, installs while you work:
%pip install -q -U langchain-core langchain-openai

In [ ]:
import json
from getpass import getpass
from openai import OpenAI

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"

# SmartMart systems — same as Day 1
INVENTORY = {"P-101": 42, "P-102": 18, "P-103": 65, "P-104": 7, "P-105": 23,
             "P-118": 3, "P-119": 14, "P-120": 73}
PRICES    = {"P-101": 1299, "P-102": 1699, "P-103": 899, "P-104": 2499,
             "P-105": 3499, "P-118": 12999, "P-119": 2799, "P-120": 599}
NAMES     = {"P-101": "Prima Electric Kettle 1.5L", "P-102": "Prima Electric Kettle 2.0L",
             "P-103": "Zenith Steel Kettle 1.0L",   "P-104": "Zenith Pro Kettle 1.8L",
             "P-105": "SwiftMix Mixer Grinder 750W","P-118": "CleanSweep Robot Vacuum",
             "P-119": "FreshBrew Coffee Maker 600ml","P-120": "ChopMaster Vegetable Chopper"}
ORDERS    = {"4521": {"status": "shipped", "eta": "tomorrow", "item": "P-105"},
             "7788": {"status": "processing", "eta": "3 days", "item": "P-101"}}

def check_inventory(product_id: str) -> str:
    if product_id not in INVENTORY: return f"Unknown product ID: {product_id}"
    return f"{INVENTORY[product_id]} units of {NAMES[product_id]} in stock"

def get_price(product_id: str) -> str:
    if product_id not in PRICES: return f"Unknown product ID: {product_id}"
    return f"{NAMES[product_id]} costs Rs. {PRICES[product_id]:,}"

def track_order(order_id: str) -> str:
    o = ORDERS.get(order_id)
    if not o: return f"No order found with ID {order_id}"
    return f"Order {order_id} ({NAMES[o['item']]}): {o['status']}, ETA {o['eta']}"

def list_products(keyword: str) -> str:
    hits = [f"{pid}: {name} (Rs. {PRICES[pid]:,})" for pid, name in NAMES.items()
            if keyword.lower() in name.lower()]
    return "\n".join(hits) if hits else f"No products matching '{keyword}'"

TOOLS = {f.__name__: f for f in [check_inventory, get_price, track_order, list_products]}
print("✅ Ready:", ", ".join(TOOLS))

---
## Rung 1 — Tools become schemas 📋

Day 1: docstrings pasted into a prompt. Today: **JSON schemas passed via `tools=`**. Same two audiences — `description` for the model, `parameters` for the machinery:

In [ ]:
def schema(name, description, params):
    """Small helper so the four schemas stay readable."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object",
                       "properties": params,
                       "required": list(params)}}}

TOOLS_SCHEMA = [
    schema("check_inventory", "How many units of a product are in stock right now.",
           {"product_id": {"type": "string", "description": "Catalog ID like P-101"}}),
    schema("get_price", "The current price in rupees of a product.",
           {"product_id": {"type": "string", "description": "Catalog ID like P-101"}}),
    schema("track_order", "Status and delivery ETA of a customer order.",
           {"order_id": {"type": "string", "description": "Order number like 4521"}}),
    schema("list_products", "Search the catalog by keyword; returns IDs, names, prices.",
           {"keyword": {"type": "string", "description": "e.g. kettle"}}),
]
print(json.dumps(TOOLS_SCHEMA[0], indent=2))   # look at one in full

---
## Rung 2 — One round trip, raw objects 🔍

Watch the **message ledger** grow through slide 5's four rows — and this time, print the actual objects the API returns:

In [ ]:
msgs = [{"role": "user", "content": "Is P-101 in stock?"}]

# ---- THINK: first API call, tools attached ----
r = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS_SCHEMA)
m = r.choices[0].message

print("The raw assistant message:")
print("  content   :", repr(m.content))
print("  tool_calls:", m.tool_calls)
tc = m.tool_calls[0]
print()
print("Look — an OBJECT, not text:")
print("  tc.id                 :", tc.id)
print("  tc.function.name      :", tc.function.name)
print("  tc.function.arguments :", tc.function.arguments, "(this bit IS still a JSON string)")

In [ ]:
# ---- ACT + OBSERVE: execute, append the two ledger rows ----
msgs.append(m)                                        # row 2: assistant + tool_calls
result = TOOLS[tc.function.name](**json.loads(tc.function.arguments))
msgs.append({"role": "tool",                          # row 3: the result, matched by id
             "tool_call_id": tc.id,
             "content": result})

# ---- THINK again: second call sees the full ledger ----
r2 = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS_SCHEMA)
print("🛒", r2.choices[0].message.content)

print("\n--- the full ledger (slide 5, live) ---")
for msg in msgs:
    role = msg["role"] if isinstance(msg, dict) else msg.role
    print(f"  [{role}]")

**Compare with Day 1:** no DECIDE prompt, no fence-stripping, no `json.loads` on prose. The one remaining `json.loads` is on `tc.function.arguments` — the API guarantees it parses.

**✏️ Your turn:** ask "track order 4521" instead — check the `tc.function.name` that comes back. The schema descriptions are doing the routing now.

---
## Rung 3 — The native loop (+ parallel calls) 🔁

Day 1's 15 lines, hardened. New superpower: **the model can request several tools in ONE turn** — the inner `for` handles it:

In [ ]:
def agent(question, msgs=None, system=None, max_steps=5, verbose=True):
    if msgs is None:
        msgs = ([{"role": "system", "content": system}] if system else []) \
               + [{"role": "user", "content": question}]
    else:
        msgs.append({"role": "user", "content": question})
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS_SCHEMA)
        m = r.choices[0].message
        if not m.tool_calls:                                     # DONE
            if verbose: print(f"  step {step} · DONE")
            msgs.append({"role": "assistant", "content": m.content})
            return m.content
        if m.content and verbose:                                # ReAct thought (rung 4)
            print(f"  step {step} · THOUGHT  → {m.content}")
        msgs.append(m)
        if verbose and len(m.tool_calls) > 1:
            print(f"  step {step} · ⚡ {len(m.tool_calls)} tool calls in ONE turn")
        for tc in m.tool_calls:                                  # ACT (maybe several)
            args = json.loads(tc.function.arguments)
            result = (TOOLS[tc.function.name](**args)
                      if tc.function.name in TOOLS else f"Unknown tool {tc.function.name}")
            if verbose: print(f"  step {step} · ACT      → {tc.function.name}({args}) → {result[:60]}")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
    return "Step limit reached."

print("🛒", agent("What do the Prima 1.5L kettle AND the robot vacuum cost?"))

**Watch for the ⚡ line** — two `get_price` calls in a single turn. Day 1's loop needed two full THINK cycles (= two extra LLM calls) for the same question. Parallel tool calls halve both latency and cost on multi-lookup questions.

**✏️ Your turn:** run Day 1's planning task — *"which kettle is cheapest, is it in stock, and what would three cost?"* — and count the LLM calls vs your Day 1 trace.

---
## Rung 4 — ReAct: thoughts in the trace 💭

One system-prompt line makes the reasoning **visible**: the assistant message carries `content` (the thought) *alongside* `tool_calls` (the action):

In [ ]:
REACT_SYSTEM = (
    "You are SmartBot's reasoning engine. Before any tool call, state your reasoning "
    "in ONE short sentence as message content alongside the tool calls. "
    "When you have enough information, answer the customer concisely.")

print("🛒", agent("A customer wants the cheapest way to make chai at home and needs it delivered fast — what should I tell them?",
                 system=REACT_SYSTEM))

**Read the THOUGHT lines** — "cheapest way to make chai → search kettles first, then compare prices…". That's ReAct: *Reason + Act*. Two wins:

1. **Better tool choice** — verbalised reasoning conditions the action (W1D2's step-by-step, agent edition)
2. **Debuggable traces** — when Friday's project misbehaves, the thought tells you *why* it picked the wrong tool

**✏️ Your turn:** run the same question WITHOUT the system prompt 3 times, then WITH it 3 times. Count how often `list_products` is (sensibly) called first. Small sample, real signal.

---
## Rung 5 — Memory across turns 🧠

The `agent()` function already accepts a `msgs` ledger — reuse it between calls and follow-ups resolve. Plus the **safe trimming rule**: a `tool_calls` message and its `role:"tool"` replies are an atomic unit — trim both or neither:

In [ ]:
ledger = [{"role": "system", "content": REACT_SYSTEM}]

print("🛒", agent("what does the 1.5L Prima kettle cost?", msgs=ledger, verbose=False))
print("🛒", agent("and is IT in stock?", msgs=ledger, verbose=False))          # 'IT' ← memory!
print("🛒", agent("okay — what did I ask you about first?", msgs=ledger, verbose=False))
print(f"\n📒 ledger now holds {len(ledger)} messages")

In [ ]:
def trim_ledger(msgs, keep_last=8):
    """Keep system + the last N messages — WITHOUT orphaning tool results.
    Rule: never let a role:'tool' message survive without its tool_calls parent."""
    system = [m for m in msgs[:1] if isinstance(m, dict) and m.get("role") == "system"]
    rest = msgs[len(system):]
    kept = rest[-keep_last:]
    # walk forward: drop any leading tool-messages whose parent got trimmed away
    while kept and (isinstance(kept[0], dict) and kept[0].get("role") == "tool"):
        kept = kept[1:]
    return system + kept

before = len(ledger)
ledger[:] = trim_ledger(ledger, keep_last=6)
print(f"trimmed {before} → {len(ledger)} messages (system preserved, no orphaned tool rows)")
print("🛒", agent("remind me the kettle price once more?", msgs=ledger, verbose=False))

**Why the orphan check matters:** send a `role:"tool"` message whose parent `tool_calls` was trimmed away and the API rejects the whole request. This is W1D4's sliding window plus one agent-specific constraint — and it's a graded item on Friday.

**✏️ Your turn:** comment out the `while` loop, trim aggressively (`keep_last=1` right after a tool call), and read the API error you get. Knowing an error on sight = never fearing it.

---
## Rung 6 — The LangChain X-ray 🩻

Same tools, same loop — through the framework. Watch every abstraction map to something you built by hand:

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI

@tool
def lc_check_inventory(product_id: str) -> str:
    """How many units of a product are in stock right now. product_id is a catalog ID like P-101."""
    return check_inventory(product_id)

@tool
def lc_get_price(product_id: str) -> str:
    """The current price in rupees of a product. product_id is a catalog ID like P-101."""
    return get_price(product_id)

# X-ray moment #1: the decorator BUILT the schema from your docstring + type hints
print("What @tool generated:")
print(json.dumps(lc_check_inventory.args_schema.model_json_schema(), indent=2)[:400])

In [ ]:
LC_TOOLS = {"lc_check_inventory": lc_check_inventory, "lc_get_price": lc_get_price}

llm = ChatOpenAI(model=MODEL, api_key=API_KEY)
agent_llm = llm.bind_tools(list(LC_TOOLS.values()))     # X-ray #2: this is tools= plumbing

def lc_agent(question, max_steps=5):
    """The same THINK/ACT/OBSERVE loop, LangChain edition — still ~15 lines."""
    msgs = [HumanMessage(question)]
    for step in range(1, max_steps + 1):
        ai = agent_llm.invoke(msgs)                     # THINK
        if not ai.tool_calls:
            return ai.content                           # DONE
        msgs.append(ai)
        for tc in ai.tool_calls:                        # ACT (same parsed objects!)
            result = LC_TOOLS[tc["name"]].invoke(tc["args"])
            print(f"  step {step} · {tc['name']}({tc['args']}) → {result[:55]}")
            msgs.append(ToolMessage(result, tool_call_id=tc["id"]))   # OBSERVE
    return "Step limit reached."

print("🛒", lc_agent("Is the Prima 1.5L kettle in stock and what does it cost?"))

**The X-ray, summarized:**

| LangChain | What you built by hand |
|---|---|
| `@tool` decorator | rung 1's schema, generated from the docstring |
| `bind_tools()` | attaching `tools=` to every call |
| `ai.tool_calls` | rung 2's parsed objects |
| `ToolMessage(..., tool_call_id=...)` | rung 2's `role:"tool"` ledger row |
| `AgentExecutor` / LangGraph's prebuilt agents | this exact loop, packaged with knobs |

Nothing in that list is magic to you anymore. **That was the point of building it raw first.**

**✏️ Your turn:** port `track_order` to LangChain (decorator + docstring) and ask about order 7788 — three lines of work.

---
## 🏁 Finale — AgentV1

In [ ]:
class AgentV1:
    """Native tool_calls · parallel execution · ReAct thoughts · memory + safe trimming."""

    def __init__(self, system=REACT_SYSTEM, max_steps=5, keep_last=10):
        self.msgs = [{"role": "system", "content": system}]
        self.max_steps = max_steps
        self.keep_last = keep_last
        self.trace = []

    def say(self, question, verbose=True):
        self.msgs.append({"role": "user", "content": question})
        for step in range(1, self.max_steps + 1):
            r = client.chat.completions.create(model=MODEL, messages=self.msgs, tools=TOOLS_SCHEMA)
            m = r.choices[0].message
            if not m.tool_calls:
                self.msgs.append({"role": "assistant", "content": m.content})
                self.trace.append(("DONE", m.content[:70]))
                self.msgs[:] = trim_ledger(self.msgs, self.keep_last)
                return m.content
            if m.content:
                self.trace.append(("THOUGHT", m.content[:70]))
                if verbose: print(f"  💭 {m.content}")
            self.msgs.append(m)
            for tc in m.tool_calls:
                args = json.loads(tc.function.arguments)
                result = (TOOLS[tc.function.name](**args)
                          if tc.function.name in TOOLS else f"Unknown tool {tc.function.name}")
                self.trace.append(("ACT", f"{tc.function.name}({args})"))
                if verbose: print(f"  🔧 {tc.function.name}({args}) → {result[:55]}")
                self.msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
        return "Step limit reached."

bot = AgentV1()
print("🛒", bot.say("I have Rs. 1,500. Which kettles can I afford, and are they in stock?"))
print()
print("🛒", bot.say("great — and how does that compare to the vacuum's price?", verbose=False))

In [ ]:
# 💬 Talk to AgentV1 — 'quit' to stop.
while True:
    user_msg = input("You: ")
    if user_msg.strip().lower() in ("quit", "exit", "q"):
        print("👋 Tomorrow: MCP — your agent learns to use OTHER PEOPLE'S tools.")
        break
    print("🛒", bot.say(user_msg))

---
## 🏆 Homework (20 min, feeds Friday's project)

1. **Reliability duel** — run *"cheapest kettle, in stock, price of three"* 5× through Day 1's `AgentV0` and 5× through `AgentV1`. Score each run: correct answer? wasted calls? malformed anything? One table, one sentence of conclusion. (This before/after is Friday demo material.)
2. **Force the tool** — add `tool_choice={"type": "function", "function": {"name": "list_products"}}` to the first call of a search-y question. When would forcing help? When would it hurt? Two sentences.
3. **Strict mode** — add `"strict": true` inside one function schema (and `"additionalProperties": false` to its parameters). Confirm it still runs, then read the docs section on structured outputs for 5 minutes — Friday's project uses this for its order-JSON tool.

### What's next
**Module 4 — MCP & multi-agent:** the Model Context Protocol — a USB port for tools, drawn with the same system interaction diagrams as Day 1 — and agents that talk to agents.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*